# Static test prediction collection

Run the frozen static Image, FEA, and Multimodal models, summarize their test results, and export the shared `static_test_predictions.csv` for downstream analyses.

Run this notebook with `4_static_facial_expression_recognition/` as the working directory, using the static TensorFlow/Keras 2.15 environment. Model files are loaded from `./models/`; the CSV is exported to this directory. Set the external dataset path below to match your Compose volume mount.

Each CSV row represents one image view. Central and side views share a `reenactment_id` and the same FEA input. FEA-native metrics use one row per reenactment; Image and Multimodal metrics use both views. Only checks needed to align inputs and predictions are retained. Results are calculated from the predictions without comparisons against manuscript values.


## 1. Paths and class order

Use the published SI images and SFEA CSV. The frozen models output classes in legacy order (`Neutral, Happiness, Sadness, Surprise, Fear, Disgust, Anger`); the exported probabilities use canonical order (`Anger, Disgust, Fear, Happiness, Neutral, Sadness, Surprise`).


In [1]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display

DATASET_ROOT = Path('/workspace/datasets')
MODEL_ROOT = Path('models')

SI_TEST_PATH = DATASET_ROOT / 'emoji-hero-vr-db-si' / 'test_set'
SFEA_TEST_PATH = DATASET_ROOT / 'emoji-hero-vr-db-sfea-as-csv' / 'test_set.csv'

IMAGE_MODEL_PATH = MODEL_ROOT / 'image_model.keras'
FEA_MODEL_PATH = MODEL_ROOT / 'fea_model.keras'
MULTIMODAL_MODEL_PATH = MODEL_ROOT / 'multimodal_model.keras'

IMAGE_SIZE = (224, 224)

OUTPUT_PATH = Path('static_test_predictions.csv')
CANONICAL_CLASSES = ['Anger', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']
CANONICAL_ID_TO_EMOTION = dict(enumerate(CANONICAL_CLASSES))
CANONICAL_EMOTION_TO_ID = {emotion: i for i, emotion in enumerate(CANONICAL_CLASSES)}
CANONICAL_TO_OLD = np.array([6, 5, 4, 1, 0, 2, 3])


2026-10-06 16:37:45.447060: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-06 16:37:45.447083: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-06 16:37:45.447739: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## 2. Load and align the test data

Image filenames identify the reenactment and camera view. Join each SFEA row to its two image views and preserve coefficient order from the source CSV, as used by the frozen models. All inference datasets follow this joined table's row order.


In [2]:
fea_df = pd.read_csv(SFEA_TEST_PATH).rename(columns={'file_id': 'reenactment_id'})
FEA_COLUMNS = [c for c in fea_df.columns if c not in {'reenactment_id', 'timestamp', 'Label'}]
assert len(FEA_COLUMNS) == 63, 'The models require 63 FEA input coefficients.'

def parse_image_path(path: Path) -> dict:
    parts = path.stem.split('-')

    if len(parts) != 7:
        raise ValueError(f'Unexpected image filename: {path.name}')

    timestamp, set_id, participant_id, level_id, emoji_id, emotion_id, camera_index = parts

    reenactment_id = '-'.join(parts[:-1])

    return {
        'sample_id': path.stem,
        'reenactment_id': reenactment_id,
        'timestamp': int(timestamp),
        'set_id': int(set_id),
        'participant_id': int(participant_id),
        'level_id': int(level_id),
        'emoji_id': int(emoji_id),
        'true_label_id': int(emotion_id),
        'camera_index': int(camera_index),
        'perspective': 'Central' if camera_index == '0' else 'Side',
        'image_path': str(path),
        'emotion_dir': path.parent.name,
    }


image_df = pd.DataFrame(parse_image_path(path) for path in SI_TEST_PATH.rglob('*.jpg'))
image_df = image_df.sort_values(['reenactment_id', 'camera_index']).reset_index(drop=True)
assert image_df['sample_id'].is_unique, 'Duplicate image-view identifiers.'
assert set(image_df['reenactment_id']) == set(fea_df['reenactment_id']), 'Image and FEA test samples do not align.'
assert image_df['emotion_dir'].map(CANONICAL_EMOTION_TO_ID).eq(image_df['true_label_id']).all(), 'Image folder and filename labels differ.'

prediction_df = image_df.merge(
    fea_df[['reenactment_id', 'Label', *FEA_COLUMNS]],
    on='reenactment_id', how='left', validate='many_to_one'
)
assert prediction_df['Label'].astype(int).eq(prediction_df['true_label_id']).all(), 'Image and FEA labels differ.'
prediction_df = prediction_df.drop(columns='Label')
prediction_df['true_label'] = prediction_df['true_label_id'].map(CANONICAL_ID_TO_EMOTION)
print(f'Loaded {len(prediction_df)} views from {prediction_df.reenactment_id.nunique()} reenactments.')


Loaded 756 views from 378 reenactments.


## 3. Inference utilities

Images retain the original JPEG decoding and resizing, with no additional normalization. Datasets are not shuffled. The multimodal dataset passes Image and FEA together as the model inputs. Probabilities and predictions are remapped to canonical class order before storage.


In [3]:
def parse_image(filename):
    image_string = tf.io.read_file(filename)
    image = tf.image.decode_jpeg(image_string, channels=3)
    image = tf.image.resize(image, IMAGE_SIZE)
    return image

def create_image_dataset(df, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices(df['image_path'].values)
    ds = ds.map(parse_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

def create_fea_dataset(df, batch_size=32):
    X = df[FEA_COLUMNS].to_numpy(dtype=np.float32)
    return tf.data.Dataset.from_tensor_slices(X).batch(batch_size)

def create_multimodal_dataset(df, batch_size=32):
    image_paths = df['image_path'].values
    feas = df[FEA_COLUMNS].to_numpy(dtype=np.float32)

    ds = tf.data.Dataset.from_tensor_slices((image_paths, feas))

    # The one-element outer tuple tells Keras that (image, fea) together form x
    # rather than interpreting the second tensor as y.
    ds = ds.map(
        lambda path, fea: ((parse_image(path), fea),),
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

def predictions_to_canonical(probabilities_old):
    probabilities_canonical = probabilities_old[:, CANONICAL_TO_OLD]
    predictions_canonical = np.argmax(probabilities_canonical, axis=1)

    return probabilities_canonical, predictions_canonical

def add_predictions(df, prefix, probabilities):
    probabilities, predictions = predictions_to_canonical(probabilities)

    df[f'{prefix}_pred_id'] = predictions
    df[f'{prefix}_pred'] = [CANONICAL_ID_TO_EMOTION[p] for p in predictions]

    emotions = [
        'Anger',
        'Disgust',
        'Fear',
        'Happiness',
        'Neutral',
        'Sadness',
        'Surprise',
    ]

    for i, emotion in enumerate(emotions):
        df[f'{prefix}_prob_{emotion.lower()}'] = probabilities[:, i]

    return predictions


## 4. Image inference


In [4]:
image_model = tf.keras.models.load_model(IMAGE_MODEL_PATH, compile=False)

print('Input:', image_model.input_shape)
print('Output:', image_model.output_shape)

image_probabilities_old = image_model.predict(
    create_image_dataset(prediction_df, batch_size=32),
    verbose=1
)

image_predictions = add_predictions(prediction_df, 'image', image_probabilities_old)

del image_model
tf.keras.backend.clear_session()
gc.collect()


Input: (None, 224, 224, 3)
Output: (None, 7)
24/24 [==============================] - 35s 8ms/step


19290

## 5. FEA inference

The same FEA vector is repeated for both views. The exported table keeps that alignment; the result summary below reports FEA metrics per unique reenactment.


In [5]:
fea_model = tf.keras.models.load_model(FEA_MODEL_PATH, compile=False)

print('Input:', fea_model.input_shape)
print('Output:', fea_model.output_shape)

fea_probabilities_old = fea_model.predict(
    create_fea_dataset(prediction_df, batch_size=32),
    verbose=1
)

fea_predictions = add_predictions(prediction_df, 'fea', fea_probabilities_old)

assert prediction_df.groupby('reenactment_id')['fea_pred_id'].nunique().eq(1).all(), 'FEA predictions differ between paired views.'

del fea_model
tf.keras.backend.clear_session()
gc.collect()


Input: (None, 63)
Output: (None, 7)
24/24 [==============================] - 0s 993us/step


1145

## 6. Multimodal inference

Supply the original custom CrossAttention layer when loading the frozen multimodal model.


In [6]:
from tensorflow.keras.layers import Layer, Dense
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras.saving import register_keras_serializable


@register_keras_serializable()
class CrossAttention(Layer):

    def __init__(self, units_v1, units_v2, **kwargs):
        super().__init__(**kwargs)
        self.units_v1 = units_v1
        self.units_v2 = units_v2

        self.weight_layer_v1 = Dense(
            units_v1,
            activation='sigmoid',
            kernel_initializer=GlorotUniform()
        )

        self.weight_layer_v2 = Dense(
            units_v2,
            activation='sigmoid',
            kernel_initializer=GlorotUniform()
        )

    def call(self, inputs, training=False):
        v1, v2 = inputs

        weights_v1 = self.weight_layer_v1(v2)
        weights_v2 = self.weight_layer_v2(v1)

        weighted_v1 = weights_v1 * v1
        weighted_v2 = weights_v2 * v2

        return [weighted_v1, weighted_v2]


In [7]:
custom_objects = {
    'CrossAttention': CrossAttention,
    'Custom>CrossAttention': CrossAttention,
}

multimodal_model = tf.keras.models.load_model(
    MULTIMODAL_MODEL_PATH,
    custom_objects=custom_objects,
    compile=False,
    safe_mode=False
)

print('Inputs:', multimodal_model.input_shape)
print('Output:', multimodal_model.output_shape)

for inp in multimodal_model.inputs:
    print(inp.name, inp.shape)

multimodal_probabilities_old = multimodal_model.predict(
    create_multimodal_dataset(prediction_df, batch_size=32),
    verbose=1
)

multimodal_predictions = add_predictions(
    prediction_df,
    'multimodal',
    multimodal_probabilities_old
)

del multimodal_model
tf.keras.backend.clear_session()
gc.collect()


Inputs: [(None, 224, 224, 3), (None, 63)]
Output: (None, 7)
input_1 (None, 224, 224, 3)
fau_input (None, 63)
24/24 [==============================] - 1s 8ms/step


4941

## 7. Test results and static complementarity

The summary gives accuracy and support-weighted F1. The class-wise table gives precision, recall, and F1 for the static baselines. FEA metrics use unique reenactments.

The correctness overlap uses paired view rows, matching multimodal evaluation. Its prediction-selection oracle counts samples where either unimodal model is correct; it is not a strict ceiling for a learned fusion model.


In [8]:
summary_rows = []
class_rows = []
for prefix in ['image', 'fea', 'multimodal']:
    frame = prediction_df.drop_duplicates('reenactment_id') if prefix == 'fea' else prediction_df
    truth = frame['true_label_id'].to_numpy(dtype=int)
    predicted = frame[f'{prefix}_pred_id'].to_numpy(dtype=int)
    cm = np.zeros((len(CANONICAL_CLASSES), len(CANONICAL_CLASSES)), dtype=int)
    np.add.at(cm, (truth, predicted), 1)
    precision = np.divide(np.diag(cm), cm.sum(axis=0), out=np.zeros(len(CANONICAL_CLASSES)), where=cm.sum(axis=0) > 0)
    recall = np.divide(np.diag(cm), cm.sum(axis=1), out=np.zeros(len(CANONICAL_CLASSES)), where=cm.sum(axis=1) > 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros(len(CANONICAL_CLASSES)), where=(precision + recall) > 0)
    summary_rows.append({'model': prefix, 'correct': int((truth == predicted).sum()), 'samples': len(frame),
                         'accuracy_percent': 100 * (truth == predicted).mean(),
                         'weighted_f1_percent': 100 * np.average(f1, weights=cm.sum(axis=1))})
    for i, category in enumerate(CANONICAL_CLASSES):
        class_rows.append({'model': prefix, 'category': category, 'support': int(cm.sum(axis=1)[i]),
                           'precision_percent': 100 * precision[i], 'recall_percent': 100 * recall[i],
                           'f1_percent': 100 * f1[i]})
static_summary = pd.DataFrame(summary_rows)
static_class_metrics = pd.DataFrame(class_rows)
display(static_summary.round(2))
display(static_class_metrics.round(2))

image_ok = prediction_df['image_pred_id'].eq(prediction_df['true_label_id'])
fea_ok = prediction_df['fea_pred_id'].eq(prediction_df['true_label_id'])
overlap = pd.DataFrame([
    {'group': 'Both correct', 'view_predictions': int((image_ok & fea_ok).sum())},
    {'group': 'FEA only correct', 'view_predictions': int((~image_ok & fea_ok).sum())},
    {'group': 'Image only correct', 'view_predictions': int((image_ok & ~fea_ok).sum())},
    {'group': 'Both incorrect', 'view_predictions': int((~image_ok & ~fea_ok).sum())},
])
display(overlap)
oracle_correct = int((image_ok | fea_ok).sum())
print(f'Prediction-selection oracle: {oracle_correct}/{len(prediction_df)} = {oracle_correct / len(prediction_df):.2%}')


,model,correct,samples,accuracy_percent,weighted_f1_percent
0,image,528,756,69.84,70.06
1,fea,271,378,71.69,71.10
2,multimodal,608,756,80.42,80.22


,model,category,support,precision_percent,recall_percent,f1_percent
0,image,Anger,108,47.41,50.93,49.11
1,image,Disgust,108,56.06,68.52,61.67
2,image,Fear,108,78.16,62.96,69.74
3,image,Happiness,108,69.92,86.11,77.18
4,image,Neutral,108,77.88,75.00,76.42
5,image,Sadness,108,80.72,62.04,70.16
6,image,Surprise,108,89.11,83.33,86.12
7,fea,Anger,54,70.27,48.15,57.14
8,fea,Disgust,54,54.10,61.11,57.39
9,fea,Fear,54,64.58,57.41,60.78


,group,view_predictions
0,Both correct,414
1,FEA only correct,128
2,Image only correct,114
3,Both incorrect,100


Prediction-selection oracle: 656/756 = 86.77%


## 8. Export shared predictions

`sample_id` is the unique view key; `reenactment_id` links central and side views. Export identifiers, predictions, and class probabilities. Raw FEA inputs are omitted. The `image_path` column records the inference source location; downstream prediction analyses use the identifiers rather than loading those paths.


In [9]:
output_columns = [
    'sample_id',
    'reenactment_id',
    'timestamp',
    'set_id',
    'participant_id',
    'level_id',
    'emoji_id',
    'camera_index',
    'perspective',
    'true_label_id',
    'true_label',
    'image_path',
]

prediction_columns = [
    c for c in prediction_df.columns
    if (
        c.startswith('image_')
        or c.startswith('fea_')
        or c.startswith('multimodal_')
    )
    and c != 'image_path'
]

output_df = prediction_df[output_columns + prediction_columns].copy()

output_df.to_csv(OUTPUT_PATH, index=False)

print(f'Exported {len(output_df)} rows to {OUTPUT_PATH}')


Exported 756 rows to static_test_predictions.csv
